# The SwiGLU feed-forward network and a mixture-of-experts (MoE) layer

Part of the free interactive course **Qwen Atlas** — chapter: [moe](https://happy-bamboo-w7dk.here.now/#/learn/moe).

```text
Run:  python 08_ffn_moe.py      (needs: numpy)
```

The output under the code cell was recorded on 2026-10-23 with Python 3.11.9, numpy 2.4.6, torch 2.14.1+cpu and transformers 5.20.2 on a CPU. Run the cell to reproduce it.


In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(4)
d, f = 8, 16

def silu(v):
    return v / (1 + np.exp(-v))

def swiglu(x, Wg, Wu, Wd):
    """Qwen3 MLP: down( SiLU(gate(x)) * up(x) ), no biases."""
    return Wd @ (silu(Wg @ x) * (Wu @ x))

x = rng.normal(size=d)
Wg, Wu = rng.normal(size=(f, d)) * 0.3, rng.normal(size=(f, d)) * 0.3
Wd = rng.normal(size=(d, f)) * 0.3
print("dense FFN output:", swiglu(x, Wg, Wu, Wd))
print("dense FFN parameters:", Wg.size + Wu.size + Wd.size)

# ---------------- MoE: a router picks top-k experts per token ----------------
E, k, fe = 8, 2, 8                       # 8 small experts, 2 used per token
experts = [(rng.normal(size=(fe, d)) * 0.3, rng.normal(size=(fe, d)) * 0.3, rng.normal(size=(d, fe)) * 0.3) for _ in range(E)]
W_router = rng.normal(size=(E, d))

logits = W_router @ x
p = np.exp(logits - logits.max()); p /= p.sum()      # softmax over all experts
top = np.argsort(-p)[:k]                             # indices of the k largest
alpha = p[top] / p[top].sum()                        # renormalize (norm_topk_prob = True in Qwen3-MoE)
y = sum(a * swiglu(x, *experts[e]) for a, e in zip(alpha, top))

print("\nrouter probabilities:", p)
print("chosen experts:", top, " mixture weights:", alpha, " sum:", alpha.sum())
print("MoE output:", y)
stored = sum(m.size for ex in experts for m in ex) + W_router.size
used = sum(m.size for e in top for m in experts[e]) + W_router.size
print(f"parameters stored: {stored}, used for this token: {used}")

dense FFN output: [ 0.5796 -0.0951 -0.0403  0.8077  0.2875  0.9787  0.0758 -0.5644]
dense FFN parameters: 384

router probabilities: [0.0143 0.049  0.0034 0.8489 0.0678 0.0054 0.0003 0.0108]
chosen experts: [3 4]  mixture weights: [0.9261 0.0739]  sum: 1.0
MoE output: [-0.656   0.8868 -0.4036  0.2622  0.0322 -0.9692  0.0747  0.231 ]
parameters stored: 1600, used for this token: 448
